In [12]:
import os, dspy
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [13]:
def utworz_lm(temperature=0.7, cache=True):
    # fragment funkcji dla backendu Ollama
    return dspy.LM(
        "ollama_chat/llama3.1",
        api_base="http://localhost:11434/v1",
        temperature=temperature,
        cache=cache,
    )


lm = utworz_lm(temperature=0.7)

In [14]:
dspy.configure(lm=lm)
print(lm("Czym jest inzynieria promptow?")[0])

Inżynieria promtów to stosowanie techniki generacji tekstu w celu automatyzacji i optymalizacji wyższych poziomów abstrakcji w procesie tworzenia kodu. Jest to technologia rozwijana przez firmy takie jak Meta.


In [15]:
class BasicQA(dspy.Signature):
    "Odpowiadaj zwięźle, krótkimi odpowiedziami."

    pytanie = dspy.InputField()
    odpowiedz = dspy.OutputField(desc="1-5 slow")

In [16]:
qa = dspy.Predict(BasicQA)

In [17]:
print(qa(pytanie="W ktorym roku powstal framework Helion DataFlow?").odpowiedz)

2019


In [18]:
KORPUS = [
    "Helion DataFlow to framework do przetwarzania strumieniowego, stworzony w 2021 roku przez Marie Kowalczyk...",
    "Jezyk zapytan Helion DataFlow nazywa sie HQL...",
    # ... kolejne fakty oraz dystraktory
]

In [19]:
import re

STOP = {
    "w",
    "i",
    "z",
    "o",
    "a",
    "do",
    "to",
    "na",
    "sie",
    "jest",
    "jego",
    "przez",
    "oraz",
    "dla",
    "od",
    "za",
    "po",
    "czy",
    "jak",
    "kto",
    "ktorym",
    "glownym",
    "ma",
    "jakim",
    "jakimi",
}


def tokeny(t):
    return {w for w in re.findall(r"\w+", t.lower()) if w not in STOP}


def retrieve(pytanie, k=4):
    q = tokeny(pytanie)
    return sorted(KORPUS, key=lambda f: len(q & tokeny(f)), reverse=True)[:k]

In [20]:
class GenerateAnswer(dspy.Signature):
    """Odpowiadaj wylacznie na podstawie kontekstu."""

    kontekst = dspy.InputField()
    pytanie = dspy.InputField()


odpowiedz = dspy.OutputField(desc="1-5 slow")

In [21]:
class RAG(dspy.Module):
    def __init__(self, k=4):
        super().__init__()
        self.k = k
        self.generuj = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, pytanie):
        ctx = retrieve(pytanie, k=self.k)
        return self.generuj(kontekst=ctx, pytanie=pytanie)

In [ ]:
class GenerateAnswer(dspy.Signature):
    """Odpowiadaj wylacznie na podstawie kontekstu."""

    kontekst: list[str] = dspy.InputField()
    pytanie: str = dspy.InputField()
    odpowiedz: str = dspy.OutputField()


print(GenerateAnswer.output_fields.keys())  # powinno zawierać 'odpowiedz'


class RAG(dspy.Module):
    def __init__(self, k=4):
        super().__init__()
        self.k = k
        self.generuj = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, pytanie):
        ctx = retrieve(pytanie, k=self.k)
        return self.generuj(kontekst=ctx, pytanie=pytanie)


rag = RAG()
wynik = rag(pytanie="W ktorym roku powstal framework Helion DataFlow?")
print(wynik.odpowiedz)

dict_keys(['odpowiedz'])
2021


In [28]:
trening = [
    dspy.Example(
        pytanie="W ktorym roku powstal Helion DataFlow?", odpowiedz="2021"
    ).with_inputs("pytanie"),
    dspy.Example(
        pytanie="Kto jest autorem Helion DataFlow?", odpowiedz="Maria Kowalczyk"
    ).with_inputs("pytanie"),
    # ... kolejne przyklady
]

In [29]:
def metryka(example, pred, trace=None):
    return example.odpowiedz.lower() in (pred.odpowiedz or "").lower()

In [30]:
from dspy.teleprompt import BootstrapFewShot

opt = BootstrapFewShot(metric=metryka, max_bootstrapped_demos=4)
rag_opt = opt.compile(RAG(), trainset=trening)
print(rag_opt(pytanie="W ktorym roku powstal Helion DataFlow?").odpowiedz)
dspy.inspect_history(n=1)

100%|██████████| 2/2 [00:09<00:00,  4.87s/it]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
2021




[2026-11-10T19:00:44.923526]

System message:

Your input fields are:
1. `kontekst` (list[str]): 
2. `pytanie` (str):
Your output fields are:
1. `plan` (str): A short plan for solving the task.
2. `odpowiedz` (str):
Output format (replace placeholders):
<plan>
{plan}
</plan>

<odpowiedz>
{odpowiedz}
</odpowiedz>
In adhering to this structure, your objective is: 
        Odpowiadaj wylacznie na podstawie kontekstu.


User message:

This is an example of the task, though some input or output fields are not supplied.

<pytanie>
Kto jest autorem Helion DataFlow?
</pytanie>


Assistant message:

<plan>
Not supplied for this particular example. 
</plan>

<odpowiedz>
Maria Kowalczyk
</odpowiedz>


User message:

<kontekst>
["Helion DataFlow to framework do przetwarzania strumieniowego, stworzony w 2021 roku przez Marie Kowalczyk...", "Jezyk zapytan Helion DataFlow nazywa sie HQL..."]
</kontekst>


In [31]:
class GenerateAnswer(dspy.Signature):
    """Zawsze odpowiadaj PELNYM ZDANIEM. Zacznij od 'Odpowiedz na to pytanie brzmi:'
    i szczegolowo wyjasnij swoja odpowiedz na podstawie kontekstu. Nigdy nie podawaj samej wartosci - zawsze rozbuduj wypowiedz."""

    kontekst = dspy.InputField()
    pytanie = dspy.InputField()
    odpowiedz = dspy.OutputField()


devset = [
    dspy.Example(
        pytanie="Jakiego jezyka zapytan uzywa Helion DataFlow?", odpowiedz="HQL"
    ).with_inputs("pytanie"),
    dspy.Example(
        pytanie="Czym zostal napisany Helion DataFlow?", odpowiedz="Scala"
    ).with_inputs("pytanie"),
    dspy.Example(
        pytanie="Kiedy powstal Helion DataFlow?", odpowiedz="2021"
    ).with_inputs("pytanie"),
    dspy.Example(
        pytanie="Kto stworzyl Helion DataFlow?", odpowiedz="Maria Kowalczyk"
    ).with_inputs("pytanie"),
]

In [32]:
import re


def _norm(s):
    s = re.sub(r"[^\w\s]", " ", (s or "").lower())
    return re.sub(r"\s+", " ", s).strip()


def metryka(example, pred, trace=None):
    return _norm(pred.odpowiedz) == _norm(example.odpowiedz)

In [34]:
from dspy.teleprompt import BootstrapFewShot

wariant_A = RAG()  # bazowy

opt = BootstrapFewShot(metric=metryka, max_bootstrapped_demos=4)
wariant_B = opt.compile(RAG(), trainset=trening)  # zoptymalizowany

100%|██████████| 2/2 [00:41<00:00, 20.65s/it]

Bootstrapped 0 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.


In [35]:
ewaluator = dspy.Evaluate(
    devset=devset, metric=metryka, num_threads=1, display_progress=True
)
lm_pomiar = utworz_lm(temperature=0.0)


def procent(program):
    with dspy.context(lm=lm_pomiar):  # wymuszenie temperature=0 na czas pomiaru
        w = ewaluator(program)
    return getattr(w, "score", w)


wynik_A = procent(wariant_A)
wynik_B = procent(wariant_B)
print(f"Wariant A (bazowy): {wynik_A} % | Wariant B (zoptymalizowany): {wynik_B} %")

Average Metric: 1.00 / 4 (25.0%): 100%|██████████| 4/4 [01:34<00:00, 23.71s/it] 

2026/11/10 19:06:59 INFO dspy.evaluate.evaluate: Average Metric: 1 / 4 (25.0%)



Average Metric: 1.00 / 4 (25.0%): 100%|██████████| 4/4 [02:05<00:00, 31.42s/it]

2026/11/10 19:09:04 INFO dspy.evaluate.evaluate: Average Metric: 1 / 4 (25.0%)



Wariant A (bazowy): 25.0 % | Wariant B (zoptymalizowany): 25.0 %


In [36]:
roznica = wynik_B - wynik_A
if wynik_B > wynik_A:
    zwyciezca = "B (zoptymalizowany)"
elif wynik_A > wynik_B:
    zwyciezca = "A (bazowy)"
else:
    zwyciezca = "remis"
print(f"Roznica B-A: {roznica:+.1f} p.p. -> zwyciezca: {zwyciezca}")

Roznica B-A: +0.0 p.p. -> zwyciezca: remis


In [37]:
POWTORZENIA = 3
for t in (0.0, 1.0):
    lm_t = utworz_lm(temperature=t, cache=(t == 0.0))
    powt = 1 if t == 0.0 else POWTORZENIA
    wyniki = []
    with dspy.context(lm=lm_t):
        for _ in range(powt):
            w = ewaluator(wariant_B)
            wyniki.append(getattr(w, "score", w))
    srednia = sum(wyniki) / len(wyniki)
    print(f"temperatura {t}: srednia {srednia:.1f} % (z {powt} przebiegow: {wyniki})")

Average Metric: 1.00 / 4 (25.0%): 100%|██████████| 4/4 [00:00<00:00, 31.14it/s]

2026/11/10 19:09:14 INFO dspy.evaluate.evaluate: Average Metric: 1 / 4 (25.0%)



temperatura 0.0: srednia 25.0 % (z 1 przebiegow: [25.0])
Average Metric: 1.00 / 4 (25.0%): 100%|██████████| 4/4 [02:24<00:00, 36.04s/it]

2026/11/10 19:11:38 INFO dspy.evaluate.evaluate: Average Metric: 1 / 4 (25.0%)



Average Metric: 0.00 / 4 (0.0%): 100%|██████████| 4/4 [01:23<00:00, 20.93s/it]

2026/11/10 19:13:02 INFO dspy.evaluate.evaluate: Average Metric: 0 / 4 (0.0%)



Average Metric: 0.00 / 4 (0.0%): 100%|██████████| 4/4 [01:20<00:00, 20.10s/it]

2026/11/10 19:14:22 INFO dspy.evaluate.evaluate: Average Metric: 0 / 4 (0.0%)



temperatura 1.0: srednia 8.3 % (z 3 przebiegow: [25.0, 0.0, 0.0])
